In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv, BatchNorm

# 1. Load Dataset
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# 2. Setup NeighborLoader
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. Optimized Architecture (No MLP Head)
class GraphTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        
        # Internal layer dropout (0.5) added directly to TransformerConv
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        
        # Layer 2: Directly to output classes with internal dropout
        self.conv2 = TransformerConv(hidden_channels * heads, out_channels, heads=heads, concat=False, dropout=0.5)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        # Higher forward dropout (0.5)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        return x

# 4. Initialize with Stronger Regularization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphTransformer(
    in_channels=dataset.num_node_features,
    hidden_channels=32, # Reduced capacity
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
# Higher weight decay (5e-3) helps keep weights small
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

# 5. Training with Label Smoothing
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        
        # label_smoothing=0.1 helps prevent overfitting to noisy labels
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size], label_smoothing=0.1)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop with Best Model Checkpointing
print(f"Starting Optimized No-MLP Training on {device}...")
best_val_acc = 0.0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'graphTransformer_noMLP_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Best: {best_val_acc:.4f}')

print(f"\nFinal Best Validation Accuracy: {best_val_acc:.4f}")

Starting Optimized No-MLP Training on cuda...
Epoch: 010, Loss: 1.5030, Val Acc: 0.5080, Best: 0.5083
Epoch: 020, Loss: 1.5035, Val Acc: 0.4819, Best: 0.5128
Epoch: 030, Loss: 1.4847, Val Acc: 0.5105, Best: 0.5128
Epoch: 040, Loss: 1.4823, Val Acc: 0.5134, Best: 0.5157
Epoch: 050, Loss: 1.4890, Val Acc: 0.4967, Best: 0.5157
Epoch: 060, Loss: 1.4709, Val Acc: 0.5154, Best: 0.5164
Epoch: 070, Loss: 1.4547, Val Acc: 0.5165, Best: 0.5165
Epoch: 080, Loss: 1.4510, Val Acc: 0.5156, Best: 0.5172
Epoch: 090, Loss: 1.4481, Val Acc: 0.5194, Best: 0.5194
Epoch: 100, Loss: 1.4382, Val Acc: 0.5141, Best: 0.5200
Epoch: 110, Loss: 1.4337, Val Acc: 0.5211, Best: 0.5211
Epoch: 120, Loss: 1.4255, Val Acc: 0.5179, Best: 0.5211
Epoch: 130, Loss: 1.4214, Val Acc: 0.5156, Best: 0.5211
Epoch: 140, Loss: 1.4180, Val Acc: 0.5199, Best: 0.5211
Epoch: 150, Loss: 1.4168, Val Acc: 0.5213, Best: 0.5213
Epoch: 160, Loss: 1.4148, Val Acc: 0.5201, Best: 0.5213
Epoch: 170, Loss: 1.4150, Val Acc: 0.5211, Best: 0.5217
Ep

## Graph Transformer - WITH MLP head

In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv, BatchNorm

# 1. Load Dataset with Laplacian Positional Encodings
# k=8 adds 8 dimensions of structural info to each node
transform = T.Compose([
    T.AddLaplacianEigenvectorPE(k=8, attr_name='pos_enc'),
    T.RandomNodeSplit(num_val=0.1, num_test=0.2)
])

dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# Concatenate original features (500) with Positional Encodings (8)
# New input dimension = 508
data.x = torch.cat([data.x, data.pos_enc], dim=-1)

# 2. Setup NeighborLoader
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. Optimized Architecture: Transformer + Residuals + MLP Head
class AdvancedTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
        super().__init__()
        
        # Layer 1: Encoder with internal dropout
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)
        # Residual connection to match dimensions for the first addition
        self.res1 = torch.nn.Linear(in_channels, hidden_channels * heads)
        
        # Layer 2: Deep aggregation
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)
        # Residual connection for the second layer
        self.res2 = torch.nn.Linear(hidden_channels * heads, hidden_channels)

        # MLP Task Head: High dropout for regularization
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.7),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        # Layer 1: Convolution + Residual
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Layer 2: Convolution + Residual
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)
        
        # Final Task Head
        x = self.mlp(x)
        return x

# 4. Initialization for Local GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# in_channels is now data.x.size(-1) which is 508
model = AdvancedTransformer(
    in_channels=data.x.size(-1), 
    hidden_channels=48, # Slightly increased capacity for PE
    out_channels=dataset.num_classes,
    heads=4
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)

# 5. Training with Label Smoothing
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        
        # Label smoothing helps generalize on noisy Flickr categories
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size], label_smoothing=0.1)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop with Best Model Checkpointing
print(f"Starting Training: Graph Transformer + PE + Residuals on {device}...")
best_val_acc = 0.0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'graphTransformer_PE_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f}, Best: {best_val_acc:.4f}')

print(f"\nOptimization Complete! Best Val Acc achieved: {best_val_acc:.4f}")

Starting Training: Graph Transformer + PE + Residuals on cuda...
Epoch: 010, Loss: 1.5585, Val Acc: 0.4974, Best: 0.5075
Epoch: 020, Loss: 1.5539, Val Acc: 0.4466, Best: 0.5098
Epoch: 030, Loss: 1.5272, Val Acc: 0.5117, Best: 0.5124
Epoch: 040, Loss: 1.5068, Val Acc: 0.5078, Best: 0.5166
Epoch: 050, Loss: 1.5008, Val Acc: 0.5086, Best: 0.5178
Epoch: 060, Loss: 1.5009, Val Acc: 0.5162, Best: 0.5187
Epoch: 070, Loss: 1.4699, Val Acc: 0.5190, Best: 0.5237
Epoch: 080, Loss: 1.4425, Val Acc: 0.5225, Best: 0.5253
Epoch: 090, Loss: 1.4146, Val Acc: 0.5208, Best: 0.5253
Epoch: 100, Loss: 1.3997, Val Acc: 0.5222, Best: 0.5253
Epoch: 110, Loss: 1.3970, Val Acc: 0.5228, Best: 0.5253
Epoch: 120, Loss: 1.3929, Val Acc: 0.5210, Best: 0.5253
Epoch: 130, Loss: 1.3902, Val Acc: 0.5225, Best: 0.5253
Epoch: 140, Loss: 1.3905, Val Acc: 0.5227, Best: 0.5253
Epoch: 150, Loss: 1.3880, Val Acc: 0.5226, Best: 0.5253
Epoch: 160, Loss: 1.3897, Val Acc: 0.5228, Best: 0.5253
Epoch: 170, Loss: 1.3889, Val Acc: 0.52

### Graph Transform WITH Residual Head

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv, BatchNorm

# ──────────────────────────────────────────────────────────────────────
# 1. Load Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Setup NeighborLoader
# ──────────────────────────────────────────────────────────────────────
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# ──────────────────────────────────────────────────────────────────────
# 3. Residual Dense Head
#
# Inspired by DenseNet: every dense block receives the *concatenation*
# of all previous block outputs (+ the original encoder embedding).
# A final projection then maps the accumulated features to class logits.
#
# Dense connections let the classifier reuse low-level encoder features
# and alleviate vanishing gradients inside the head itself, while the
# residual (skip) path from the encoder output ensures that the raw
# GNN embedding is never lost.
# ──────────────────────────────────────────────────────────────────────
class ResidualDenseHead(nn.Module):
    """
    A 3-block dense head with residual skip connections.

    Given encoder output h (dim = in_dim):
        Block 1:  d1 = ReLU(BN(Linear(h)))                  -> dense_dim
        Block 2:  d2 = ReLU(BN(Linear(h || d1)))             -> dense_dim
        Block 3:  d3 = ReLU(BN(Linear(h || d1 || d2)))       -> dense_dim
        Output :  logits = Linear(h || d1 || d2 || d3)       -> out_dim

    '||' denotes concatenation.
    """
    def __init__(self, in_dim, dense_dim, out_dim, drop_p=0.5):
        super().__init__()

        # Dense block 1: takes encoder output only
        self.fc1 = nn.Linear(in_dim, dense_dim)
        self.bn1 = nn.BatchNorm1d(dense_dim)

        # Dense block 2: takes encoder output + block 1 output
        self.fc2 = nn.Linear(in_dim + dense_dim, dense_dim)
        self.bn2 = nn.BatchNorm1d(dense_dim)

        # Dense block 3: takes encoder output + blocks 1 & 2
        self.fc3 = nn.Linear(in_dim + dense_dim * 2, dense_dim)
        self.bn3 = nn.BatchNorm1d(dense_dim)

        # Final projection: all concatenated features -> classes
        self.out_proj = nn.Linear(in_dim + dense_dim * 3, out_dim)

        self.drop_p = drop_p

    def forward(self, h):
        # h: [N, in_dim]  (encoder output)

        d1 = F.relu(self.bn1(self.fc1(h)))
        d1 = F.dropout(d1, p=self.drop_p, training=self.training)

        cat1 = torch.cat([h, d1], dim=-1)                    # [N, in_dim + dense_dim]
        d2 = F.relu(self.bn2(self.fc2(cat1)))
        d2 = F.dropout(d2, p=self.drop_p, training=self.training)

        cat2 = torch.cat([h, d1, d2], dim=-1)                # [N, in_dim + 2*dense_dim]
        d3 = F.relu(self.bn3(self.fc3(cat2)))
        d3 = F.dropout(d3, p=self.drop_p, training=self.training)

        cat3 = torch.cat([h, d1, d2, d3], dim=-1)            # [N, in_dim + 3*dense_dim]
        return self.out_proj(cat3)


# ──────────────────────────────────────────────────────────────────────
# 4. GraphTransformer Encoder + Residual Dense Head
# ──────────────────────────────────────────────────────────────────────
class GraphTransformerWithDenseHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, dense_dim=64, head_drop=0.5):
        super().__init__()

        # ── Encoder (same 2-layer TransformerConv backbone) ──
        self.conv1 = TransformerConv(in_channels, hidden_channels,
                                     heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)

        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels,
                                     heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)

        # Residual skip in the encoder (project input to match conv1 output)
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)
        # Residual skip for conv2 (project conv1 output to match conv2 output)
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)

        # ── Residual Dense Classification Head ──
        self.head = ResidualDenseHead(
            in_dim=hidden_channels,
            dense_dim=dense_dim,
            out_dim=out_channels,
            drop_p=head_drop,
        )

    def forward(self, x, edge_index):
        # --- Encoder block 1 ---
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # --- Encoder block 2 ---
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)

        # --- Residual Dense Head ---
        return self.head(x)


# ──────────────────────────────────────────────────────────────────────
# 5. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GraphTransformerWithDenseHead(
    in_channels=dataset.num_node_features,   # 500
    hidden_channels=32,                      # encoder bottleneck
    out_channels=dataset.num_classes,        # 7
    heads=4,
    dense_dim=64,                            # width of each dense block
    head_drop=0.5,
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10
)


# ──────────────────────────────────────────────────────────────────────
# 6. Training with Label Smoothing
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size],
            label_smoothing=0.1,
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


# ──────────────────────────────────────────────────────────────────────
# 7. Execution Loop with Best Model Checkpointing
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GraphTransformer + Residual Dense Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_acc = 0.0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'graphTransformer_denseHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_acc:.4f}, Val: {val_acc:.4f}, '
              f'Test: {test_acc:.4f}, Best Val: {best_val_acc:.4f}')

print(f"\nFinal Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy at Best Val:      {best_test_acc:.4f}")

Starting GraphTransformer + Residual Dense Head Training on cuda...
Model parameters: 399,335
Epoch: 010, Loss: 1.5076, Train: 0.5262, Val: 0.5075, Test: 0.5098, Best Val: 0.5075
Epoch: 020, Loss: 1.4995, Train: 0.5310, Val: 0.5150, Test: 0.5157, Best Val: 0.5150
Epoch: 030, Loss: 1.5085, Train: 0.5035, Val: 0.4904, Test: 0.4893, Best Val: 0.5244
Epoch: 040, Loss: 1.4784, Train: 0.5450, Val: 0.5203, Test: 0.5197, Best Val: 0.5272
Epoch: 050, Loss: 1.4560, Train: 0.5461, Val: 0.5204, Test: 0.5193, Best Val: 0.5272
Epoch: 060, Loss: 1.4334, Train: 0.5722, Val: 0.5286, Test: 0.5227, Best Val: 0.5339
Epoch: 070, Loss: 1.4048, Train: 0.5828, Val: 0.5211, Test: 0.5170, Best Val: 0.5339
Epoch: 080, Loss: 1.3710, Train: 0.6105, Val: 0.5330, Test: 0.5230, Best Val: 0.5339
Epoch: 090, Loss: 1.3459, Train: 0.6278, Val: 0.5319, Test: 0.5245, Best Val: 0.5339
Epoch: 100, Loss: 1.3280, Train: 0.6391, Val: 0.5269, Test: 0.5213, Best Val: 0.5339
Epoch: 110, Loss: 1.3193, Train: 0.6433, Val: 0.5290, Te

### Graph Transform WITH Gated Head

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv, BatchNorm


# ──────────────────────────────────────────────────────────────────────
# 1. Load Dataset
# ──────────────────────────────────────────────────────────────────────
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# ──────────────────────────────────────────────────────────────────────
# 2. Setup NeighborLoader
# ──────────────────────────────────────────────────────────────────────
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)


# ──────────────────────────────────────────────────────────────────────
# 3. Gated Head
#
# Inspired by LSTM / GRU / Highway Networks.
# Each GatedBlock computes:
#     transform = ReLU(BN(Linear(x)))       — candidate new features
#     gate      = Sigmoid(Linear(x))        — importance scores in [0, 1]
#     output    = gate * transform + (1 - gate) * x
#
# The gate learns to let useful features through and suppress noise.
# Stacking multiple gated blocks gives the model several chances to
# refine the embedding before the final classification projection.
# ──────────────────────────────────────────────────────────────────────
class GatedBlock(nn.Module):
    """Single gated layer: learns which features to keep vs. transform."""
    def __init__(self, dim, drop_p=0.5):
        super().__init__()
        # Transform path: produces candidate replacement features
        self.fc_transform = nn.Linear(dim, dim)
        self.bn = nn.BatchNorm1d(dim)

        # Gate path: produces per-feature importance scores
        self.fc_gate = nn.Linear(dim, dim)

        self.drop_p = drop_p

    def forward(self, x):
        # Candidate features
        t = F.relu(self.bn(self.fc_transform(x)))
        t = F.dropout(t, p=self.drop_p, training=self.training)

        # Gate: sigmoid squashes to [0, 1] per feature
        g = torch.sigmoid(self.fc_gate(x))

        # Highway-style combination:
        #   g ≈ 1  →  use transformed features (learned representation)
        #   g ≈ 0  →  pass through original features (skip connection)
        return g * t + (1 - g) * x


class GatedHead(nn.Module):
    """
    Multi-layer gated classification head.

    Encoder output h (dim = in_dim)
        → Linear projection to gate_dim (if in_dim != gate_dim)
        → GatedBlock × num_blocks   (each refines features)
        → Linear projection to out_dim (class logits)
    """
    def __init__(self, in_dim, gate_dim, out_dim, num_blocks=3, drop_p=0.5):
        super().__init__()

        # Project encoder embedding to the gating dimension
        self.input_proj = nn.Linear(in_dim, gate_dim) if in_dim != gate_dim else nn.Identity()

        # Stack of gated blocks — each refines the representation
        self.blocks = nn.ModuleList([
            GatedBlock(gate_dim, drop_p=drop_p) for _ in range(num_blocks)
        ])

        # Final classification projection
        self.out_proj = nn.Linear(gate_dim, out_dim)

    def forward(self, h):
        x = self.input_proj(h)
        for block in self.blocks:
            x = block(x)
        return self.out_proj(x)


# ──────────────────────────────────────────────────────────────────────
# 4. GraphTransformer Encoder + Gated Head
# ──────────────────────────────────────────────────────────────────────
class GraphTransformerWithGatedHead(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, gate_dim=128, num_gate_blocks=3, head_drop=0.5):
        super().__init__()

        # ── Encoder (same 2-layer TransformerConv backbone) ──
        self.conv1 = TransformerConv(in_channels, hidden_channels,
                                     heads=heads, dropout=0.5)
        self.bn1 = BatchNorm(hidden_channels * heads)

        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels,
                                     heads=heads, concat=False, dropout=0.5)
        self.bn2 = BatchNorm(hidden_channels)

        # Residual skips in the encoder
        self.res1 = nn.Linear(in_channels, hidden_channels * heads)
        self.res2 = nn.Linear(hidden_channels * heads, hidden_channels)

        # ── Gated Classification Head ──
        self.head = GatedHead(
            in_dim=hidden_channels,
            gate_dim=gate_dim,
            out_dim=out_channels,
            num_blocks=num_gate_blocks,
            drop_p=head_drop,
        )

    def forward(self, x, edge_index):
        # --- Encoder block 1 ---
        identity = self.res1(x)
        x = self.conv1(x, edge_index) + identity
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # --- Encoder block 2 ---
        identity = self.res2(x)
        x = self.conv2(x, edge_index) + identity
        x = self.bn2(x)
        x = F.relu(x)

        # --- Gated Head ---
        return self.head(x)


# ──────────────────────────────────────────────────────────────────────
# 5. Initialization
# ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GraphTransformerWithGatedHead(
    in_channels=dataset.num_node_features,   # 500
    hidden_channels=32,                      # encoder bottleneck
    out_channels=dataset.num_classes,        # 7
    heads=4,
    gate_dim=128,                            # width of gating blocks
    num_gate_blocks=3,                       # 3 rounds of gated refinement
    head_drop=0.5,
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10
)


# ──────────────────────────────────────────────────────────────────────
# 6. Training with Label Smoothing
# ──────────────────────────────────────────────────────────────────────
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)

        loss = F.cross_entropy(
            out[:batch.batch_size],
            batch.y[:batch.batch_size],
            label_smoothing=0.1,
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs


# ──────────────────────────────────────────────────────────────────────
# 7. Execution Loop with Best Model Checkpointing
# ──────────────────────────────────────────────────────────────────────
print(f"Starting GraphTransformer + Gated Head Training on {device}...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
best_val_acc = 0.0
best_test_acc = 0.0

for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc
        torch.save(model.state_dict(), 'graphTransformer_gatedHead_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train: {train_acc:.4f}, Val: {val_acc:.4f}, '
              f'Test: {test_acc:.4f}, Best Val: {best_val_acc:.4f}')

print(f"\nFinal Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy at Best Val:      {best_test_acc:.4f}")


Starting GraphTransformer + Gated Head Training on cuda...
Model parameters: 483,719
Epoch: 010, Loss: 1.5241, Train: 0.5213, Val: 0.5192, Test: 0.5069, Best Val: 0.5192
Epoch: 020, Loss: 1.5199, Train: 0.5235, Val: 0.5234, Test: 0.5105, Best Val: 0.5234
Epoch: 030, Loss: 1.5285, Train: 0.5261, Val: 0.5207, Test: 0.5123, Best Val: 0.5234
Epoch: 040, Loss: 1.5050, Train: 0.5362, Val: 0.5245, Test: 0.5137, Best Val: 0.5245
Epoch: 050, Loss: 1.5042, Train: 0.5347, Val: 0.5229, Test: 0.5146, Best Val: 0.5281
Epoch: 060, Loss: 1.4820, Train: 0.5512, Val: 0.5313, Test: 0.5202, Best Val: 0.5313
Epoch: 070, Loss: 1.4548, Train: 0.5713, Val: 0.5309, Test: 0.5235, Best Val: 0.5365
Epoch: 080, Loss: 1.4451, Train: 0.5777, Val: 0.5317, Test: 0.5240, Best Val: 0.5365
Epoch: 090, Loss: 1.4223, Train: 0.6051, Val: 0.5345, Test: 0.5253, Best Val: 0.5365
Epoch: 100, Loss: 1.3887, Train: 0.6200, Val: 0.5334, Test: 0.5251, Best Val: 0.5365
Epoch: 110, Loss: 1.3730, Train: 0.6335, Val: 0.5294, Test: 0.523

## MORE optimizations -  3 layer graph transform , 

In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv, LayerNorm

# 1. Dataset with higher-dimensional PE
# Increasing k to 16 gives more granular structural information
transform = T.Compose([
    T.AddLaplacianEigenvectorPE(k=16, attr_name='pos_enc'),
    T.RandomNodeSplit(num_val=0.1, num_test=0.2)
])

dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]
data.x = torch.cat([data.x, data.pos_enc], dim=-1)

# 2. NeighborLoader (Larger batch for stability)
train_loader = NeighborLoader(
    data,
    num_neighbors=[20, 15, 10], # One sample size per layer
    batch_size=2048, 
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. Deep & Wide Architecture
class DeepTransformer(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        
        # Layer 1
        self.conv1 = TransformerConv(in_channels, hidden_channels, heads=heads, dropout=0.4)
        self.ln1 = LayerNorm(hidden_channels * heads)
        self.res1 = torch.nn.Linear(in_channels, hidden_channels * heads)
        
        # Layer 2
        self.conv2 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=0.4)
        self.ln2 = LayerNorm(hidden_channels * heads)
        self.res2 = torch.nn.Linear(hidden_channels * heads, hidden_channels * heads)

        # Layer 3 (Concat=False to average heads for the MLP input)
        self.conv3 = TransformerConv(hidden_channels * heads, hidden_channels, heads=heads, concat=False, dropout=0.4)
        self.ln3 = LayerNorm(hidden_channels)
        self.res3 = torch.nn.Linear(hidden_channels * heads, hidden_channels)

        # Wider MLP Head
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels * 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels * 2, out_channels)
        )

    def forward(self, x, edge_index):
        # Block 1
        x = F.relu(self.ln1(self.conv1(x, edge_index) + self.res1(x)))
        
        # Block 2
        x = F.relu(self.ln2(self.conv2(x, edge_index) + self.res2(x)))
        
        # Block 3
        x = F.relu(self.ln3(self.conv3(x, edge_index) + self.res3(x)))
        
        return self.mlp(x)

# 4. Initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepTransformer(
    in_channels=data.x.size(-1), 
    hidden_channels=32, # 32 * 8 heads = 256 dims per layer
    out_channels=dataset.num_classes,
    heads=8
).to(device)

data = data.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)

# Use Cosine Annealing for a smoother descent
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# 5. Training Logic
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size], label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    accs = [ (pred[mask] == data.y[mask]).sum().item() / mask.sum().item() 
            for mask in [data.train_mask, data.val_mask, data.test_mask] ]
    return accs

# 6. Execution Loop
best_val_acc = 0
for epoch in range(1, 201):
    loss = train()
    train_acc, val_acc, test_acc = test()
    scheduler.step() # Step every epoch
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'transformer_ultra_best.pt')

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {val_acc:.4f}, Best: {best_val_acc:.4f}')

Epoch: 010, Loss: 1.4653, Val: 0.5257, Best: 0.5257
Epoch: 020, Loss: 1.2137, Val: 0.5068, Best: 0.5286


KeyboardInterrupt: 